# IQM demo

In [2]:
%load_ext autoreload
%autoreload 2

In [ ]:
from py4lexis.session import LexisSession

lexis_session = LexisSession()

In [ ]:
from qaas import QProvider, QBackend

token = lexis_session.get_access_token()
lexis_project = ...
resource_name = ...

provider = QProvider(token, lexis_project)
# QBackend
backend:QBackend = provider.get_backend(resource_name)

print(f'Qubit: {backend.architecture.qubits}')

print(f'Gates: {backend.architecture.gates.keys()}')

Qubit: ['QB1', 'QB2', 'QB3', 'QB4', 'QB5', 'QB6', 'QB7', 'QB8', 'QB9', 'QB10', 'QB11', 'QB12', 'QB13', 'QB14', 'QB15', 'QB16', 'QB17', 'QB18', 'QB19', 'QB20', 'QB21', 'QB22', 'QB23', 'QB24']
Gates: dict_keys(['measure', 'measure_fidelity', 'move', 'cz', 'prx', 'prx_12', 'cc_prx', 'reset_wait'])


### Solver usage

In [5]:
from QHyper.problems.maxcut import MaxCutProblem
from QHyper.util import sort_solver_results

problem = MaxCutProblem(edges=[(0, 1)])

In [6]:
from QHyper.solvers.gate_based.iqm import QAOA
from QHyper.optimizers import OptimizationParameter
from QHyper.optimizers.scipy_minimizer import ScipyOptimizer

solver = QAOA(problem,
    layers=2,
    gamma=OptimizationParameter(min=[0.0, 0.0],
                                init=[0.5, 0.5],
                                max=[6.28, 6.28]),
    beta=OptimizationParameter(min=[0.0, 0.0],
                            init=[1.0, 1.0],
                            max=[6.28, 6.28]),
    optimizer=ScipyOptimizer(verbose=True, method='Powell'),
    penalty_weights=[1],
    use_simulator=True,

    shots=1000
)

iqm_results = solver.solve()

C:\Users\Kacper\Documents\QHyper\QHyper\optimizers\scipy_minimizer.py:97: OptimizeWarning: Unknown solver options: maxfun
  result = scipy.optimize.minimize(


Step 1/200: -1.0
Step 2/200: -1.0
Step 3/200: -1.0
Success: True. Message: Optimization terminated successfully.


In [7]:
sorted_results = sort_solver_results(
    iqm_results.probabilities)

print(sorted_results.dtype.names)
for result in sorted_results:
    print(result)

('x0', 'x1', 'probability')
(0, 1, 0.504)
(1, 0, 0.494)
(1, 1, 0.001)
(0, 0, 0.001)


In [12]:
iqm_results.params

{'gamma': array([1.51850865, 0.14695309]),
 'beta': array([1.37300951, 1.36122847])}

In [13]:
solver = QAOA(problem,
    layers=2,
    gamma=OptimizationParameter(init=[1.51850865, 0.14695309]),
    beta=OptimizationParameter(init=[1.37300951, 1.36122847]),
    penalty_weights=[1],
    use_lexis=True,
    lexis_project=lexis_project,
    lexis_resource_name=resource_name,
    lexis_token=token,

    shots=1000
)

lexis_results = solver.solve()

In [14]:
sorted_results = sort_solver_results(
    lexis_results.probabilities)

print(sorted_results.dtype.names)
for result in sorted_results:
    print(result)

('x0', 'x1', 'probability')
(1, 0, 0.482)
(0, 1, 0.275)
(0, 0, 0.179)
(1, 1, 0.064)
